In [1]:
import sys
!{sys.executable} -m pip install --quiet tf-keras
!{sys.executable} -m pip install --quiet nltk
!{sys.executable} -m pip install --upgrade --quiet ipywidgets
!{sys.executable} -m pip install --quiet bertopic
!{sys.executable} -m pip install --quiet ipywidgets
!{sys.executable} -m pip install --quiet nbformat
!{sys.executable} -m pip install --quiet pysentimiento
!{sys.executable} -m pip install --quiet pyarrow==21.0
!{sys.executable} -m pip install --quiet --upgrade numpy
!{sys.executable} -m pip install --quiet wordcloud


In [10]:
import json
import nltk
import re
from nltk.corpus import stopwords
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from hdbscan import HDBSCAN
from umap import UMAP
from IPython.display import display
import pandas as pd
import os
from tqdm import tqdm
from sklearn.feature_extraction.text import CountVectorizer

classif = pd.read_json('../gpt-4o_results.json', lines=True)
classif = classif.rename(columns={'id' : 'video_id'})
classif = classif[classif['video_id'] != '2V5KZJgi_WA']
classif_yes = classif[classif['classification'] == 'sim'].reset_index(drop=True)
classif_yes.head()

,video_id,classification,title,description
0,hsKeK6zD7ho,sim,PARTE 41: VACINA TRÍPLICE VIRAL #enfermagemcon...,"acina sarampo, caxumba, rubéola (atenuada) – S..."
1,ZWkJuIAo-g0,sim,Vacina do HPV é só para meninas?,Muitos ainda acreditam que a vacina contra o H...
2,RvDQEbrQ780,sim,Vacina causa intoxicação por alumínio? O que a...,💉 Vacinas causam intoxicação por alumínio? A r...
3,LjpImqFOhZE,sim,PARTE 38: VACINA INFLUENZA TRIVALENTE #enferm...,"Vacina influenza trivalente (fragmentada, inat..."
4,nup1H9bSWwY,sim,PARTE 42: VACINA TRÍPLICE VIRAL #enfermagemcon...,"Vacina sarampo, caxumba, rubéola (atenuada) – ..."


In [11]:
def limpar_texto_func(texto, stop_words):
    if not isinstance(texto, str): return ""
    texto = texto.lower()
    # 1. Lowercase
    texto = texto.lower()
    
    # 2. Remover URLs
    texto = re.sub(r"http\S+", "", texto)
    
    # 3. **Compressão de caracteres repetidos** (kkkkkkk → kkk)
    texto = re.sub(r"(.)\1{2,}", r"\1\1\1", texto)
    
    # 4. Remover emojis e símbolos (mantém letras acentuadas)
    texto = re.sub(r"[^a-zA-ZÀ-ÿ\s]", "", texto)
    palavras = texto.split()
    palavras = [p for p in palavras if p not in stop_words and len(p) > 2]
    return " ".join(palavras)


In [14]:
def executar_analise_bertopic(df, limpar_texto_func, params):

    etapas = [
        "Baixando stopwords",
        "Limpando textos",
        "Criando vectorizer",
        "Criando UMAP",
        "Criando HDBSCAN",
        "Executando BERTopic",
        "Gerando resultados"
    ]

    pbar = tqdm(total=len(etapas), desc="Execução do modelo", ncols=100)

    # === STOPWORDS ===
    nltk.download('stopwords', quiet=True)
    stop_words = set(stopwords.words("portuguese"))
    stop_words.update(["pra", "vc", "pq", "porque", "tbm", "sobre", "kkk"])
    pbar.update(1)

    # === LIMPEZA ===
    mensagens_limpas = []
    mensagens_originais = []

    for _, row in df.iterrows():
        try:
            conteudo = row["title"].strip() + '\n' + row['description'].strip()
            if conteudo:
                txt = limpar_texto_func(conteudo, stop_words)
                if txt:
                    mensagens_limpas.append(txt)
                    mensagens_originais.append(conteudo)
        except:
            continue

    pbar.update(1)

    # === MODELOS ===
    vectorizer_model = CountVectorizer(
        ngram_range=params["ngram_range"],
        stop_words=list(stop_words)
    )
    pbar.update(1)

    umap_model = UMAP(
        n_neighbors=params["n_neighbors"],
        n_components=params["n_components"],
        min_dist=params["min_dist"],
        metric='cosine',
        random_state=42
    )
    pbar.update(1)

    hdbscan_model = HDBSCAN(
        min_cluster_size=params["min_cluster_size"],
        min_samples=params["min_samples"],
        cluster_selection_epsilon=params["cluster_selection_epsilon"],
        metric='euclidean',
        cluster_selection_method='eom'
    )
    pbar.update(1)

    # === BERTopic ===
    topic_model = BERTopic(
        language="portuguese",
        vectorizer_model=vectorizer_model,
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        min_topic_size=params["min_topic_size"],
        verbose=True
    )

    topics, probs = topic_model.fit_transform(mensagens_limpas)
    pbar.update(1)

    df_info = topic_model.get_topic_info()
    doc_info = topic_model.get_document_info(mensagens_limpas)
    doc_info["texto_original"] = mensagens_originais
    pbar.update(1)

    pbar.close()

    return {
        "topic_model": topic_model,
        "df_info": df_info,
        "topics": topics,
        "probs": probs,
        "doc_info": doc_info,
    }



# ==========================
#     EXECUÇÃO EM LOTE
# ==========================

def rodar_experimentos(df, limpar_texto_func):

    # Diretório de saída
    OUTPUT_DIR = "resultados_bertopic_descricao"
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # Listas de parâmetros para testar
    param_grid = {
        "ngram_range": [(1,1), (1, 2)],
        "n_neighbors": [15],
        "n_components": [5],
        "min_dist": [0.0],
        "min_cluster_size": [100],
        "min_samples": [15],
        "cluster_selection_epsilon": [0.5, 0.75],
        "min_topic_size": [100],
    }

    # Geração do espaço de busca (lista de dicionários)
    from itertools import product

    keys, values = zip(*param_grid.items())
    param_combinations = [dict(zip(keys, v)) for v in product(*values)]

    print(f"Total de combinações de parâmetros: {len(param_combinations)}")

    # Loop com tqdm
    for i, params in enumerate(tqdm(param_combinations, desc="Rodando experimentos")):
        
        print("\n-----------------------------------------")
        print(f"Execução {i+1}/{len(param_combinations)}")
        print("Parâmetros:", params)

        resultados = executar_analise_bertopic(df, limpar_texto_func, params)

        # Diretório da execução
        run_dir = os.path.join(OUTPUT_DIR, f"execucao_{i:03d}")
        os.makedirs(run_dir, exist_ok=True)

        # === salvar df_info ===
        resultados["df_info"].to_csv(os.path.join(run_dir, "df_info.csv"), index=False)

        # === salvar doc_info ===
        resultados["doc_info"].to_csv(os.path.join(run_dir, "doc_info.csv"), index=False)

        # === salvar parâmetros ===
        with open(os.path.join(run_dir, "parametros.json"), "w", encoding="utf-8") as f:
            json.dump(params, f, indent=4, ensure_ascii=False)

        # === salvar modelo ===
        resultados["topic_model"].save(os.path.join(run_dir, "modelo_bertopic"))

        # Log simples
        with open(os.path.join(run_dir, "log.txt"), "w", encoding="utf-8") as f:
            f.write(str(resultados["df_info"].head()))


    print("\n\n>>> EXPERIMENTOS CONCLUÍDOS COM SUCESSO! <<<")
    print(f"Resultados salvos em: {OUTPUT_DIR}")


In [15]:
rodar_experimentos(classif_yes, limpar_texto_func)

Total de combinações de parâmetros: 4


Rodando experimentos:   0%|          | 0/4 [00:00<?, ?it/s]


-----------------------------------------
Execução 1/4
Parâmetros: {'ngram_range': (1, 1), 'n_neighbors': 15, 'n_components': 5, 'min_dist': 0.0, 'min_cluster_size': 100, 'min_samples': 15, 'cluster_selection_epsilon': 0.5, 'min_topic_size': 100}



Execução do modelo:  29%|████████████▊                                | 2/7 [00:01<00:03,  1.31it/s]2025-12-08 16:34:55,448 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/468 [00:00<?, ?it/s]

...


-----------------------------------------
Execução 4/4
Parâmetros: {'ngram_range': (1, 2), 'n_neighbors': 15, 'n_components': 5, 'min_dist': 0.0, 'min_cluster_size': 100, 'min_samples': 15, 'cluster_selection_epsilon': 0.75, 'min_topic_size': 100}



Execução do modelo:  29%|████████████▊                                | 2/7 [00:02<00:05,  1.08s/it]2025-12-08 16:50:31,067 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/468 [00:00<?, ?it/s]

2025-12-08 17:12:29,120 - BERTopic - Embedding - Completed ✓
2025-12-08 17:12:29,123 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-12-08 17:12:47,597 - BERTopic - Dimensionality - Completed ✓
2025-12-08 17:12:47,598 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-12-08 17:12:48,170 - BERTopic - Cluster - Completed ✓
2025-12-08 17:12:48,176 - BERTopic - Representation - Fine-tuning topics using representation models.
2025-12-08 17:12:51,183 - BERTopic - Representation - Completed ✓

Execução do modelo: 100%|████████████████████████████████████████████| 7/7 [22:22<00:00, 191.83s/it]
2025-12-08 17:12:52,941 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.
Rodando experimentos: 100%|██████████| 4/4 [38:07<00:00, 571.95s/it]



>>> EXPERIMENTOS CONCLUÍDOS COM SUCESSO! <<<
Resultados salvos em: resultados_bertopic_descricao


In [106]:
# DataFrame de documentos com probabilidades
doc_info = doc_info_filtered  # já contém 'Topic', 'Probability' e 'texto_original'

top_docs_por_topico = []

# Itera sobre todos os tópicos únicos, ignorando o -1
for topico_id in sorted(doc_info['Topic'].unique()):
    if topico_id == -1:
        continue
    
    # Pega os 10 documentos mais prováveis para o tópico
    docs_topico = doc_info[doc_info.Topic == topico_id] \
        .sort_values(by='Probability', ascending=False) \
        .head(10)['texto_original'].tolist()
    
    # Adiciona cada doc junto com o tópico na lista
    for doc in docs_topico:
        top_docs_por_topico.append({
            'topic_id': topico_id,
            'comentário': doc
        })

# Converte para DataFrame
df_topicos = pd.DataFrame(top_docs_por_topico)

# Salva em CSV
df_topicos.to_csv('mensagens_por_topico.csv', index=False, encoding='utf-8-sig')

print(f"CSV gerado com {len(df_topicos)} mensagens representativas.")


CSV gerado com 210 mensagens representativas.
